In [ ]:
# %pip install soundfile
# !pip install -q kaggle
# !pip install -q wandb
# !pip install -q python-
!pip install -q pesq pystoi

## Libraries Declaration

In [ ]:
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple, Iterator, Sequence
from pathlib import Path
import json
import re
import wave
import numpy as np
import soundfile as sf
import os

import torch
from torch import Tensor, nn
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import BatchSampler
import time

from collections import defaultdict
import matplotlib.pyplot as plt
import random

## Mount Google Drive to Colab 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Download dataset
- The dataset is stored in the temporary Colab filesystem
- The /content directory is from Colab. This code is run in Colab
- The commented commands are to download echo dataset from AEC challenge
- The current dataset is from DNS challenge

In [ ]:
%%bash

# %cd /content

# !git clone --filter=blob:none --no-checkout \
#     https://github.com/microsoft/AEC-Challenge.git

# %cd /content/AEC-Challenge

# !git sparse-checkout init --cone

# !git sparse-checkout set \
#     datasets/synthetic/nearend_mic_signal \
#     datasets/synthetic/nearend_speech

# !git checkout main

# RAW_DIR="/content/drive/MyDrive/DNS_Data/raw"
# cd "$RAW_DIR"

# echo "1/2 Downloading clean speech chunk (4.66 GB) to Google Drive..."
# curl -L -C - -o clean_speech_000.tar.bz2 \
#   "https://dns4public.blob.core.windows.net/dns4archive/datasets_fullband/clean_fullband/datasets_fullband.clean_fullband.read_speech_000_0.00_3.75.tar.bz2"

# echo "2/2 Downloading noise chunk (5.36 GB) to Google Drive..."
# curl -L -C - -o noise_audioset_000.tar.bz2 \
#   "https://dns4public.blob.core.windows.net/dns4archive/datasets_fullband/noise_fullband/datasets_fullband.noise_fullband.audioset_000.tar.bz2"

# echo "Raw archives saved in Google Drive:"
# ls -lh "$RAW_DIR"


In [ ]:
# %%bash
# RAW_DIR="/content/drive/MyDrive/DNS_Data/raw"
# mkdir -p /content/dns_scratch/clean
# mkdir -p /content/dns_scratch/noise

# echo "Extracting clean speech to local scratch..."
# tar -xjf "$RAW_DIR/clean_speech_000.tar.bz2" -C /content/dns_scratch/clean/

# echo "Extracting noise to local scratch..."
# tar -xjf "$RAW_DIR/noise_audioset_000.tar.bz2" -C /content/dns_scratch/noise/

# echo "Local extraction complete!"

In [ ]:
# %%bash
# cd /content/dns_scratch

# echo "Unpacking clean speech .tar files..."
# for f in clean/*.tar clean/*/*.tar; do
#   [ -f "$f" ] && tar -xf "$f" -C clean/ && rm -f "$f"
# done

# echo "Unpacking noise .tar files..."
# for f in noise/*.tar noise/*/*.tar; do
#   [ -f "$f" ] && tar -xf "$f" -C noise/ && rm -f "$f"
# done

# echo "Done! Verifying .wav files:"
# find clean/ -name "*.wav" | wc -l
# find noise/ -name "*.wav" | wc -l

In [ ]:
# import os
# import subprocess
# import tarfile
# from pathlib import Path

# # Google Drive paths
# gdrive_raw = Path('/content/drive/MyDrive/DNS_Data/raw')
# clean_dest = gdrive_raw / 'clean'
# noise_dest = gdrive_raw / 'noise'

# clean_dest.mkdir(parents=True, exist_ok=True)
# noise_dest.mkdir(parents=True, exist_ok=True)

# # 1. Step 1: Unpack outer .tar.bz2 archives into clean/ and noise/
# print("="*60)
# print("STEP 1: Extracting outer .tar.bz2 archives in Google Drive...")
# print("="*60)

# clean_bz2 = gdrive_raw / 'clean_speech_000.tar.bz2'
# noise_bz2 = gdrive_raw / 'noise_audioset_000.tar.bz2'

# if clean_bz2.exists():
#     print(f"Extracting {clean_bz2.name} into {clean_dest} ...")
#     subprocess.run(['tar', '-xjf', str(clean_bz2), '-C', str(clean_dest)], check=True)
#     print("Clean speech outer archive extracted!")
# else:
#     print(f"{clean_bz2.name} not found, checking existing files...")

# if noise_bz2.exists():
#     print(f"\nExtracting {noise_bz2.name} into {noise_dest} ...")
#     subprocess.run(['tar', '-xjf', str(noise_bz2), '-C', str(noise_dest)], check=True)
#     print("Noise outer archive extracted!")
# else:
#     print(f"{noise_bz2.name} not found, checking existing files...")

# # 2. Step 2: Unpack any inner .tar files found inside clean/ and noise/
# print("\n" + "="*60)
# print("STEP 2: Unpacking inner .tar files to extract .wav files...")
# print("="*60)

# for category, target_dir in [("clean", clean_dest), ("noise", noise_dest)]:
#     # Find all nested .tar archives
#     tar_files = list(target_dir.rglob("*.tar")) + list(target_dir.rglob("*.tar.gz")) + list(target_dir.rglob("*.tgz"))
#     print(f"\n[{category.upper()}] Found {len(tar_files)} inner .tar archive(s) in Google Drive:")
    
#     for idx, tf in enumerate(tar_files, 1):
#         print(f" [{idx}/{len(tar_files)}] Unpacking {tf.name}...")
#         try:
#             with tarfile.open(tf, "r:*") as archive:
#                 archive.extractall(path=target_dir)
#             # Remove the inner .tar file to save space on Google Drive
#             tf.unlink()
#             print(f"      Done and removed {tf.name}.")
#         except Exception as e:
#             print(f"      Error extracting {tf.name}: {e}")

# # 3. Step 3: Verify the extracted .wav files on Google Drive
# print("\n" + "="*60)
# print("VERIFICATION ON GOOGLE DRIVE:")
# print("="*60)

# clean_wavs = list(clean_dest.rglob("*.wav"))
# noise_wavs = list(noise_dest.rglob("*.wav"))

# print(f"Total Clean Speech WAV files on Google Drive: {len(clean_wavs)}")
# print(f"Total Noise Audio WAV files on Google Drive:   {len(noise_wavs)}")

# if clean_wavs:
#     print(f"\nSample clean WAV: {clean_wavs[0]}")
# if noise_wavs:
#     print(f"Sample noise WAV: {noise_wavs[0]}")

## Synthesize clean speech and noise to create a noisy speech
This script generates synthetic, paired noisy and clean audio samples from raw datasets (e.g., DNS Challenge) for training speech enhancement models.

### 1. Key Steps:
Audio Discovery & Setup: Scans raw directories for clean speech and noise .wav files and prepares target output directories (synthesized/noisy and synthesized/clean).
### 2. Standardization:
- Downmixes multi-channel audio to mono.
- Resamples audio to a target sample rate of 16 kHz.
- Crops or pads/loops each clip to an exact duration of 10 seconds (160,000 samples).
### 3. SNR Mixing:
- Randomly samples a Signal-to-Noise Ratio (SNR) between -5 dB and +20 dB.
- Dynamically scales noise energy relative to speech power before mixing.
- ### Noisy_speech = Clean_Speech + scale * Audio_Noise
### 4. Clipping Prevention: 
Normalizes amplitudes to keep peak signal levels at or below 0.95 (avoiding digital clipping/distortion).
### 5. Aligned Export: 
Saves paired files matching the naming scheme fileid_{i}.wav for direct compatibility with indexed audio loaders (such as IndexedERBDataset).

In [ ]:
wav, sr = torchaudio.load("/content/drive/MyDrive/DNS_Data/raw/clean/datasets_fullband/clean_fullband/read_speech/book_00000_chp_0009_reader_06709_13_seg_2.wav")
print(sr)

In [ ]:
ROOT_OUT = Path("/content/drive/MyDrive/DNS_Data/synthesized")
RAW_CLEAN = Path("/content/drive/MyDrive/DNS_Data/raw/clean")
RAW_NOISE = Path("/content/drive/MyDrive/DNS_Data/raw/noise")

TRAIN_SIZE = 2000
VAL_SIZE = 200
TEST_SIZE = 200

SNR_LEVELS = [-5, 0, 5, 10, 15, 20]   # dB

TARGET_SR = 16000  # Hz
DURATION_SEC = 10  # seconds
TARGET_SAMPLES = TARGET_SR * DURATION_SEC  # 160,000 samples/frames in each audio

SEED = 42

In [ ]:
def process_audio(wav: torch.Tensor, sr: int, target_sr: int, target_samples: int, is_noise: bool = False) -> torch.Tensor:
    """Mono-mix -> resample -> crop/pad to exactly target_samples."""
    # Mix mono
    wav = wav.mean(dim=0)

    # Resample
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    # Crop/pad to target length
    # numel() in Pytorch stands for number of elements in a Tensor
    if wav.numel() < target_samples:
        if is_noise:
            # Loop noise until long enough
            repeats = (target_samples // wav.numel()) + 1
            wav = wav.repeat(repeats)[:target_samples]
        else:
            # Pad speech with silence
            wav = F.pad(wav, (0, target_samples - wav.numel()))
    else:
        start = random.randint(0, wav.numel() - target_samples)
        wav = wav[start : start + target_samples]

    return wav

def mix_at_snr(clean: torch.Tensor, noise: torch.Tensor, snr_db: float) -> tuple[torch.Tensor, torch.Tensor]:
    speech_power = clean.pow(2).mean().clamp_min(1e-8)
    noise_power = noise.pow(2).mean().clamp_min(1e-8)
    scale = torch.sqrt(speech_power / noise_power / (10 ** (snr_db / 10.0)))
    noisy = clean + scale * noise

    # Prevent clipping — apply the same gain to both signals
    peak = max(noisy.abs().max(), clean.abs().max())
    if peak > 1.0:
        factor = 0.95 / peak
        noisy = noisy * factor
        clean_save = clean * factor
    else:
        clean_save = clean

    return noisy, clean_save

def synthesize_split(split_name: str, clean_paths: list[Path], noise_files: list[Path], start_idx: int):
    out_noisy = ROOT_OUT / split_name / "noisy"
    out_clean = ROOT_OUT / split_name / "clean"
    out_noisy.mkdir(parents=True, exist_ok=True)
    out_clean.mkdir(parents=True, exist_ok=True)

    total_pairs = len(clean_paths) * len(SNR_LEVELS)
    print(os.linesep + f"[{split_name}] {len(clean_paths)} clean files x "
          f"{len(SNR_LEVELS)} SNRs = {total_pairs} pairs")

    for i, clean_path in enumerate(clean_paths):
        clean_wav, c_sr = torchaudio.load(clean_path)
        clean_wav = process_audio(clean_wav, c_sr, TARGET_SR, TARGET_SAMPLES, is_noise=False)

        for snr_db in SNR_LEVELS:
            noise_wav, n_sr = torchaudio.load(random.choice(noise_files))
            noise_wav = process_audio(noise_wav, n_sr, TARGET_SR, TARGET_SAMPLES, is_noise=True)

            noisy_wav, clean_save = mix_at_snr(clean_wav, noise_wav, snr_db)

            fname = f"fileid_{start_idx}.wav"
            torchaudio.save(str(out_noisy / fname), noisy_wav.unsqueeze(0), TARGET_SR)
            torchaudio.save(str(out_clean / fname), clean_save.unsqueeze(0), TARGET_SR)

            start_idx += 1

        if (i + 1) % 100 == 0 or (i + 1) == len(clean_paths):
            print(f"   Progress: {i + 1}/{len(clean_paths)} clean files processed", flush=True)


In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

clean_files = list(RAW_CLEAN.rglob("*.wav"))
noise_files = list(RAW_NOISE.rglob("*.wav"))

if not clean_files:
    raise FileNotFoundError(f"No clean .wav files found in {RAW_CLEAN}")
if not noise_files:
    raise FileNotFoundError(f"No noise .wav files found in {RAW_NOISE}")

total_needed = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
if len(clean_files) < total_needed:
    raise ValueError(
        f"Not enough clean files: need {total_needed}, "
        f"found {len(clean_files)}"
    )

print(f"Found {len(clean_files)} clean files and {len(noise_files)} noise files.")
print(f"Using {total_needed} clean files total "
      f"({TRAIN_SIZE} train / {VAL_SIZE} val / {TEST_SIZE} test).")

random.shuffle(clean_files)
train_files = clean_files[:TRAIN_SIZE]
val_files = clean_files[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE]
test_files = clean_files[TRAIN_SIZE + VAL_SIZE : TRAIN_SIZE + VAL_SIZE + TEST_SIZE]


synthesize_split("train", train_files, noise_files, start_idx=0)
synthesize_split("val", val_files, noise_files, start_idx=0)
synthesize_split("test", test_files, noise_files, start_idx=0)

print("\n── Dataset Summary ─────────────────────────────────────────")
total_pairs = 0
for split in ("train", "val", "test"):
    n = len(list((ROOT_OUT / split / "noisy").glob("*.wav")))
    total_pairs += n
    print(f"  {split:5s}  noisy pairs: {n}")
print(f"  {'TOTAL':5s}  noisy pairs: {total_pairs}")
print(f"\nAll files saved under: {ROOT_OUT}")


## Old Version

In [ ]:
# out_noisy = Path("/content/drive/MyDrive/DNS_Data/synthesized/noisy")
# out_clean = Path("/content/drive/MyDrive/DNS_Data/synthesized/clean")
# out_noisy.mkdir(parents=True, exist_ok=True)
# out_clean.mkdir(parents=True, exist_ok=True)

# clean_files = list(Path("/content/drive/MyDrive/DNS_Data/raw/clean").rglob("*.wav"))
# noise_files = list(Path("/content/drive/MyDrive/DNS_Data/raw/noise").rglob("*.wav"))
# if not clean_files:
#     raise FileNotFoundError("No clean .wav files found!")
# if not noise_files:
#     raise FileNotFoundError("No noise .wav files found!")
# print(f"Found {len(clean_files)} clean files and {len(noise_files)} noise files.")

# TARGET_SR     = 16000
# DURATION_SEC  = 10
# TARGET_SAMPLES = TARGET_SR * DURATION_SEC
# NUM_PAIRS     = 18000   
# print(f"Synthesizing {NUM_PAIRS} noisy/clean pairs...")
# for i in range(NUM_PAIRS):
#     if i % 200 == 0:
#         print(f"  Progress: {i}/{NUM_PAIRS}", flush=True)
#     # 1. Load random clean speech and noise files
#     clean_wav, c_sr = torchaudio.load(random.choice(clean_files))
#     noise_wav, n_sr = torchaudio.load(random.choice(noise_files))
#     # 2. Mix down to mono
#     clean_wav = clean_wav.mean(dim=0)
#     noise_wav = noise_wav.mean(dim=0)
#     # 3. Resample to 16 kHz
#     if c_sr != TARGET_SR:
#         clean_wav = torchaudio.functional.resample(clean_wav, c_sr, TARGET_SR)
#     if n_sr != TARGET_SR:
#         noise_wav = torchaudio.functional.resample(noise_wav, n_sr, TARGET_SR)
#     # 4. Crop or pad to exactly 10 seconds
#     if clean_wav.numel() < TARGET_SAMPLES:
#         clean_wav = torch.nn.functional.pad(clean_wav, (0, TARGET_SAMPLES - clean_wav.numel()))
#     else:
#         start = random.randint(0, clean_wav.numel() - TARGET_SAMPLES)
#         clean_wav = clean_wav[start : start + TARGET_SAMPLES]
#     if noise_wav.numel() < TARGET_SAMPLES:
#         noise_wav = noise_wav.repeat((TARGET_SAMPLES // noise_wav.numel()) + 1)[:TARGET_SAMPLES]
#     else:
#         start = random.randint(0, noise_wav.numel() - TARGET_SAMPLES)
#         noise_wav = noise_wav[start : start + TARGET_SAMPLES]
#     # 5. Mix at a random SNR between -5 dB and +20 dB
#     # 10, 15, 20, 15 dB
#     snr_db = random.uniform(-5.0, 20.0)
#     speech_power = clean_wav.pow(2).mean().clamp_min(1e-8)
#     noise_power  = noise_wav.pow(2).mean().clamp_min(1e-8)
#     scale = torch.sqrt(speech_power / noise_power / (10 ** (snr_db / 10.0)))
#     noisy_wav = clean_wav + scale * noise_wav
#     # 6. Prevent clipping (keep peak below 0 dBFS)
#     peak = max(noisy_wav.abs().max(), clean_wav.abs().max())
#     if peak > 1.0:
#         noisy_wav = noisy_wav / peak * 0.95
#         clean_wav = clean_wav / peak * 0.95
#     # 7. Save aligned pairs (fileid_N.wav naming matches your IndexedERBDataset)
#     filename = f"fileid_{i}.wav"
#     torchaudio.save(str(out_noisy / filename), noisy_wav.unsqueeze(0), TARGET_SR)
#     torchaudio.save(str(out_clean / filename), clean_wav.unsqueeze(0), TARGET_SR)
# print(f"\nDone! Generated {NUM_PAIRS} pairs in {out_noisy.parent}")

## Upload dataset to Kaggle

In [ ]:
!pip install --upgrade --quiet kaggle


In [ ]:
import os
import json
from pathlib import Path

# --- 1. SET YOUR CREDENTIALS ---
KAGGLE_USERNAME = "quanninhhoang"
# Paste your token here (from kaggle.json or your KGAT_ token)
KAGGLE_TOKEN = "KGAT_04fe7c39b1b7b3efc4434841a5e2a6d9".strip()

# --- 2. SETUP DIRECTORY AND FILES ---
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

# Format A: Plain text token (what the new CLI specifically looks for)
(kaggle_dir / "access_token").write_text(KAGGLE_TOKEN)
(kaggle_dir / "access_token").chmod(0o600)

# Format B: Traditional kaggle.json (for backward compatibility)
(kaggle_dir / "kaggle.json").write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_TOKEN})
)
(kaggle_dir / "kaggle.json").chmod(0o600)


# # --- 3. VERIFY AUTHENTICATION ---
# print("Testing Kaggle authentication...")
# !kaggle datasets list --max-size 1

In [ ]:
!rm -r /content/kaggle_erb_upload

In [ ]:
import json
from pathlib import Path

dataset_dir = Path("/content/drive/MyDrive/DNS_Data/synthesized")

metadata = {
    "title": "DNS-Speech-Dataset",
    "id": "quanninhhoang/dns-speech-dataset",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(dataset_dir / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f)

In [ ]:
!cd /content/drive/MyDrive/DNS_Data/synthesized && zip -r -q /content/dns_dataset.zip train val test

In [ ]:
!mkdir -p /content/kaggle_upload
!mv -f /content/dns_dataset.zip /content/kaggle_upload/
!cp /content/drive/MyDrive/DNS_Data/synthesized/dataset-metadata.json /content/kaggle_upload/

In [ ]:
!df -h /content

In [ ]:
!kaggle datasets create -p /content/kaggle_upload -u

In [ ]:
!kaggle datasets create -p /content/drive/MyDrive/DNS_Data/erb_store/ --dir-mode zip

## Verify dataset

In [ ]:
# input_dir = Path(
#     "/content/AEC-Challenge/datasets/synthetic/nearend_mic_signal"
# )

# target_dir = Path(
#     "/content/AEC-Challenge/datasets/synthetic/nearend_speech"
# )

# print(len(list(input_dir.glob("*.wav"))), "input files")
# print(len(list(target_dir.glob("*.wav"))), "target files")

In [ ]:
# GDRIVE_ROOT = Path('/content/drive/MyDrive/DNS_Data')
# GDRIVE_RAW = GDRIVE_ROOT / 'raw'
# GDRIVE_SHARDS = GDRIVE_ROOT / 'erb_store'
# GDRIVE_RAW.mkdir(parents=True, exist_ok=True)
# GDRIVE_SHARDS.mkdir(parents=True, exist_ok=True)

# # Local fast scratch space on Colab (for extracting & fast synthesis)
# LOCAL_SCRATCH = Path('/content/dns_scratch')
# LOCAL_SCRATCH.mkdir(parents=True, exist_ok=True)
# print("Directories ready on Google Drive:")
# print(" - Raw data directory:   ", GDRIVE_RAW)
# print(" - Shards directory:     ", GDRIVE_SHARDS)


## ERB filter bank

In [ ]:
wav, sr = torchaudio.load("/kaggle/input/datasets/quanninhhoang/dns-speech-dataset/train/clean/fileid_0.wav")
print(sr)

In [ ]:
def load_mono(path):
    wav, sr = torchaudio.load(path)    # sr: sampling rate
    wav = wav.mean(dim=0)
    return wav, sr

def stft_magnitude(wav, device, n_fft=512, hop_length=128, win_length=512):
    window = torch.hann_window(win_length, device=device)
    wav = wav.to(device)

    spec = torch.stft(
        wav,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        return_complex=True,
    )

    # [freq, time] -> [time, freq]
    return spec.transpose(-1, -2)

def hz_to_erb(freq):
    return 21.4 * torch.log10(1.0 + 0.00437 * freq)

def erb_to_hz(erb):
    return (10 ** (erb / 21.4) - 1.0) / 0.00437

def make_erb_filterbank(sample_rate, n_fft, erb_bins, low_freq=0.0, high_freq=None):
    if high_freq is None:
        high_freq = sample_rate / 2

    erb_edges = torch.linspace(
        hz_to_erb(torch.tensor(low_freq)),
        hz_to_erb(torch.tensor(high_freq)),
        erb_bins + 2,
    )

    hz_edges = erb_to_hz(erb_edges)
    fft_freqs = torch.linspace(0.0, sample_rate / 2, n_fft // 2 + 1)

    filterbank = torch.zeros(erb_bins, n_fft // 2 + 1)

    for i in range(erb_bins):
        left = hz_edges[i]
        center = hz_edges[i + 1]
        right = hz_edges[i + 2]

        rising = (fft_freqs - left) / (center - left)
        falling = (right - fft_freqs) / (right - center)

        filterbank[i] = torch.minimum(rising, falling).clamp_min(0.0)
    return filterbank

def wav_to_erb(path, sample_rate, filterbank, n_fft=512, hop_length=128, win_length=512):
    wav, sr = load_mono(path)

    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=sample_rate)

    magnitude = stft_magnitude(wav, n_fft=n_fft, hop_length=hop_length, win_length=win_length)
    erb = magnitude @ filterbank.T
    erb = torch.log1p(erb)
    return erb 

def wav_pair_to_features(input_path: str | Path, target_path: str | Path, *, sample_rate: int, n_fft: int, hop_length: int, win_length: int, filterbank: Tensor, device) -> Dict[str, Tensor]:
    input_wav, input_sr = load_mono(input_path)
    target_wav, target_sr = load_mono(target_path)

    if input_sr != sample_rate:
        input_wav = torchaudio.functional.resample(input_wav, input_sr, sample_rate)
    if target_sr != sample_rate:
        target_wav = torchaudio.functional.resample(target_wav, target_sr, sample_rate)

    sample_count = min(input_wav.numel(), target_wav.numel())
    input_wav = input_wav[:sample_count]
    target_wav = target_wav[:sample_count]

    input_spec = stft_magnitude(input_wav, device, n_fft, hop_length, win_length)
    target_spec = stft_magnitude(target_wav, device, n_fft, hop_length, win_length)
    frame_count = min(input_spec.shape[0], target_spec.shape[0])
    input_spec = input_spec[:frame_count].contiguous()
    target_spec = target_spec[:frame_count].contiguous()

    input_power = input_spec.abs().square()
    target_power = target_spec.abs().square()
    input_erb = torch.log1p(input_power @ filterbank.to(device).T).float()
    target_erb = torch.log1p(target_power @ filterbank.to(device).T).float()

    record = {
        "input_erb": input_erb.cpu().half(),
        "target_erb": target_erb.cpu().half(),
        "input_spec": input_spec.to(torch.complex64).cpu(),
        "target_spec": target_spec.to(torch.complex64).cpu(),
    }

    del input_wav
    del target_wav
    del input_spec
    del target_spec
    del input_erb
    del target_erb

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return record

def build_erb_store(input_dir: str | Path, 
                    target_dir: str | Path, 
                    output_dir: str | Path, 
                    *, 
                    sample_rate: int = 16000, 
                    n_fft: int = 512, 
                    hop_length: int = 128, 
                    win_length: int = 512, 
                    erb_bins: int = 32,
                    device) -> None:
    input_dir = Path(input_dir)
    target_dir = Path(target_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    input_files = sorted(p for p in input_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".wav")
    target_files = sorted(p for p in target_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".wav")

    if not input_files:
        raise FileNotFoundError(f"No WAV files found under {input_dir}")
    if not target_files:
        raise FileNotFoundError(f"No WAV files found under {target_dir}")

    target_by_id: Dict[str, Path] = {}
    for target_path in target_files:
        match = re.search(r"(?:^|_)fileid_(.+)\.wav$", target_path.name)
        if match:
            target_by_id[match.group(1)] = target_path

    filterbank = make_erb_filterbank(sample_rate, n_fft, erb_bins)
    index_rows: List[Dict[str, Any]] = []
    shard_size = 200
    shard = []
    shard_id = 0

    for record_id, input_path in enumerate(input_files):
        if record_id % 100 == 0:
            print(
                f"Processing {record_id}/{len(input_files)}",
                flush=True,
            )
        target_path = target_dir / input_path.name
        if not target_path.exists():
            match = re.search(r"(?:^|_)fileid_(.+)\.wav$", input_path.name)
            target_path = target_by_id.get(match.group(1)) if match else None

        if target_path is None or not target_path.exists():
            raise FileNotFoundError(f"Missing target for {input_path.name} under {target_dir}")

        record = wav_pair_to_features(
            input_path,
            target_path,
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            filterbank=filterbank,
            device=device,
        )
        shard_index = len(shard)
        shard.append(record)

        index_rows.append(
            {
                "id": record_id,
                "input_file": str(input_path),
                "target_file": str(target_path),
                "shard_file": f"shard_{shard_id:04d}.pt",
                "shard_index": shard_index,
                "frames": int(record["input_erb"].shape[0]),
                "erb_bins": int(record["input_erb"].shape[1])
            }
        )

        if (len(shard) == shard_size):
            shard_path = (output_dir / f"shard_{shard_id:04d}.pt")
            torch.save(shard, shard_path)
            shard = []
            shard_id += 1
    if shard:
        shard_path = (output_dir / f"shard_{shard_id:04d}.pt")
        torch.save(shard, shard_path)
        
    index_path = output_dir / "index.jsonl"
    with index_path.open("w", encoding="utf-8") as handle:
        for row in index_rows:
            handle.write(json.dumps(row) + os.linesep)


## Loss Function

In [ ]:
"""Return normalized inverse ERB weights with shape ``[F, E]``."""
def erb_synthesis_matrix(filterbank: Tensor, eps: float = 1e-8) -> Tensor:
    return filterbank.T / filterbank.sum(dim=0, keepdim=True).T.clamp_min(eps)

def apply_erb_gains(input_spec: Tensor, gains: Tensor, synthesis_matrix: Tensor) -> Tensor:
    if gains.ndim != 4 or gains.shape[1] != 1:
        raise ValueError("gains must have shape [B, 1, T, E]")
    """synthesis_matrix: [F, E], which converts values defined on 
    ERB bands back into values for individual FFT frequency bins.
       gains[:, 0]: makes [B, 1, T, E] -> [B, T, E]"""
    frequency_gain = gains[:, 0] @ synthesis_matrix.T    # -> [B, T, F]
    return input_spec * frequency_gain.to(input_spec.dtype)

def apply_deep_filter(stage1_spec: Tensor, coefficients: Tensor, alpha: Tensor, df_bins: int = 96, df_order: int = 5, lookahead: int = 0) -> Tensor:
    B, T, F = stage1_spec.shape

    # Complex coefficients
    c = torch.view_as_complex(coefficients.contiguous())

    # Low-frequency bands of Stage 1 output (Y^G)
    yg_low = stage1_spec[:, :, :df_bins]

    # Casual/lookahead padding
    pad_front = df_order - 1 - lookahead
    pad_back = lookahead
    pad_front_tensor = torch.zeros((B, pad_front, df_bins), dtype=yg_low.dtype, device=yg_low.device)
    pad_back_tensor = torch.zeros((B, pad_back, df_bins), dtype=yg_low.dtype, device=yg_low.device) if pad_back > 0 else None

    if pad_back_tensor is not None:
        padded = torch.cat([pad_front_tensor, yg_low, pad_back_tensor], dim=1)
    else:
        padded = torch.cat([pad_front_tensor, yg_low], dim=1)

    # Gather delayed frames: tap n corresponds to lag n
    frames = torch.stack([padded[:, pad_front - n : pad_front - n + T, :] for n in range(df_order)], dim=-1)  # [B, T, df_bins, df_order=5]

    # Filtered spectrogram Y^DF0
    y_df0 = (c * frames).sum(dim=-1)  # [B, T, df_bins]

    # Convex combination with alpha: α · Y^DF0 + (1 - α) · Y^G
    y_df = alpha * y_df0 + (1.0 - alpha) * yg_low

    # Full spectrum: low-frequency bins enhanced, high-frequency bins pass through from Y^G
    final_spec = stage1_spec.clone()
    final_spec[:, :, :df_bins] = y_df

    return final_spec

### This is a derivative of arctan(b, a), where z = a + bi
$$
\delta X = \delta \phi \cdot \left( \frac{-\Im\{X\}}{|X_h|^2}, \frac{\Re\{X\}}{|X_h|^2} \right)
$$

In [ ]:
class SafeAngle(torch.autograd.Function):
    def forward(ctx, x: Tensor) -> Tensor:
        ctx.save_for_backward(x)
        return torch.atan2(x.imag, x.real)

    def backward(ctx, grad: Tensor) -> Tensor:
        (x,) = ctx.saved_tensors
        # |X_h|^2 = max(Re^2 + Im^2, 1e-12)
        denom = (x.real.square() + x.imag.square()).clamp_min(1e-12)
        grad_scaled = grad / denom
        return torch.view_as_complex(torch.stack([-x.imag * grad_scaled, x.real * grad_scaled], dim=-1))

safe_angle = SafeAngle.apply

### 1. Compressed Spectral Loss and ERB Mask Loss

In [ ]:
class EnhancedSpectralLoss(nn.Module):
    def __init__(self, power: float = 0.6, factor_magnitude: float = 1.0, factor_complex: float = 1.0, factor_under: float = 2.0):
        super().__init__()
        self.power = power
        self.factor_magnitude = factor_magnitude
        self.factor_complex = factor_complex
        self.factor_under = factor_under

    def forward(self, predicted_spec: Tensor, target_spec: Tensor):
        pred_mag = predicted_spec.abs().clamp_min(1e-8).pow(self.power)
        target_mag = target_spec.abs().clamp_min(1e-8).pow(self.power)

        mag_err = (pred_mag - target_mag).square()
        if self.factor_under != 1.0:
            # Overpenalize when prediction < target (speech was cut)
            under_weight = torch.where(pred_mag < target_mag, self.factor_under, 1)
            mag_err = mag_err * under_weight

        loss_mag = mag_err.mean() * self.factor_magnitude

        loss_complex = torch.tensor(0.0, device=predicted_spec.device)
        if self.factor_complex > 0:
            pred_c = pred_mag * torch.exp(1j * safe_angle(predicted_spec))
            target_c = target_mag * torch.exp(1j * safe_angle(target_spec))
            loss_complex = F.mse_loss(torch.view_as_real(pred_c), torch.view_as_real(target_c)) * self.factor_complex

        return loss_mag + loss_complex

class ERBMaskLoss(nn.Module):
    """
    Direct supervision on the ERB mask using Ideal Amplitude Mask (IAM)
    Compute the loss of output gains with ideal gains/IAM
    Uses L2 + L4 penalties to heavily penalize large gain errors
    """
    def __init__(self, power_compress: float = 0.6, factor_under: float = 2.0, factor_l2: float = 1.0, factor_l4: float = 10.0, eps: float = 1e-12):
        super().__init__()
        self.power_compress = power_compress
        self.factor_under = factor_under
        self.factor_l2 = factor_l2
        self.factor_l4 = factor_l4
        self.eps = eps

    def forward(self, predicted_gains: Tensor, input_spec: Tensor, target_spec: Tensor, filterbank: Tensor) -> Tensor:
        input_pow = input_spec.abs().square()
        target_pow = target_spec.abs().square()

        fb = filterbank.to(input_spec.device).T
        input_erb_pow = input_pow @ fb
        target_erb_pow = target_pow @ fb

        # Ideal Amplitude Mask in ERB domain: sqrt(S_power / (X_power + eps))
        target_mask = (target_erb_pow / (input_erb_pow + self.eps)).sqrt().clamp(0.0, 1.0)
        target_mask = target_mask.unsqueeze(1) # [B, 1, T, E]

        # Power compress target and prediction for dynamic range equalization
        g_pred = predicted_gains.clamp_min(self.eps).pow(self.power_compress)
        g_true = target_mask.clamp_min(self.eps).pow(self.power_compress)

        diff = g_pred - g_true
        if self.factor_under != 1.0:
            diff = diff * torch.where(g_pred < g_true, self.factor_under, 1.0)

        # L2 and L4 norm penalty
        loss_l2 = diff.square().clamp_min(1e-13).mean() * self.factor_l2
        loss_l4 = diff.square().clamp_min(1e-13).pow(2).mean() * self.factor_l4

        return loss_l2 + loss_l4

### 2. Local SNR Calculation and Alpha Loss

In [ ]:
class LocalSnrTarget(nn.Module):
    """Compute Ground-Truth Local SNR in low-frequency bins over 20ms"""
    def __init__(self, df_bins: int = 96, window_size: int = 5):
        super().__init__()
        self.df_bins = df_bins
        self.window_size = window_size

    def forward(self, clean_spec: Tensor, noisy_spec: Tensor) -> Tensor:
        noise_spec = noisy_spec - clean_spec

        # Sum powers in DF region: [B, T]
        clean_pow = clean_spec[:, :, :self.df_bins].abs().square().sum(dim=-1)
        noise_pow = noise_spec[:, :, :self.df_bins].abs().square().sum(dim=-1)

        # 20 ms moving average smoothing across time (kernel_size ~ 5 frames)
        ws = self.window_size
        clean_pow = F.avg_pool1d(clean_pow.unsqueeze(1), kernel_size=ws, stride=1, padding=ws // 2).squeeze(1)
        noise_pow = F.avg_pool1d(noise_pow.unsqueeze(1), kernel_size=ws, stride=1, padding=ws // 2).squeeze(1)

        lsnr_db = 10.0 * torch.log10(clean_pow.clamp_min(1e-12) / noise_pow.clamp_min(1e-12))
        return lsnr_db  # [B, T]

class DfAlphaLoss(nn.Module):
    def __init__(self, factor: float = 0.05, lsnr_thresh: float = -7.5, lsnr_min: float = -10.0):
        super().__init__()
        self.factor = factor
        self.lsnr_thresh = lsnr_thresh
        self.lsnr_min = lsnr_min
        
    def lsnr_mapping(self, lsnr: Tensor, thresh_high: float, thresh_low: float) -> Tensor:
        slope = 1.0 / (thresh_high - thresh_low)
        return 1.0 - torch.clamp((lsnr - thresh_low) * slope, 0.0, 1.0)

    def forward(self, pred_alpha: Tensor, target_lsnr: Tensor) -> Tensor:
        # pred_alpha: [B, T, 1], target_lsnr: [B, T]
        target_lsnr = target_lsnr.unsqueeze(-1)

        # 1. Penalize alpha > 0 when LSNR < -10 dB (force alpha -> 0)
        w_off = self.lsnr_mapping(target_lsnr, self.lsnr_thresh, self.lsnr_min)
        l_off = (pred_alpha * w_off).square().mean()

        # 2. Penalize alpha < 1 when LSNR > -5 dB (encourage DF activation)
        w_on = 1.0 - self.lsnr_mapping(target_lsnr, 0.0, self.lsnr_thresh + 2.5)
        l_on = 0.1 * ((1.0 - pred_alpha) * w_on).abs().mean()

        return (l_off + l_on) * self.factor
        
        
class MultiResSpecLoss(nn.Module):
    """Compute compressed spectral loss across multiple STFT resolutions"""
    def __init__(self, n_ffts: List[int] = [256, 512, 1024], gamma: float = 0.6, factor_mag: float = 1.0, factor_complex: float = 1.0):
        super().__init__()
        self.gamma = gamma
        self.n_ffts = n_ffts
        self.factor_mag = factor_mag
        self.factor_complex = factor_complex

    def forward(self, pred_wav: Tensor, target_wav: Tensor) -> Tensor:
        device = pred_wav.device
        loss = torch.zeros((), device=device)

        for n_fft in self.n_ffts:
            hop_length = n_fft // 4
            win_length = n_fft

            y = stft_magnitude(pred_wav, device=device, n_fft=n_fft, hop_length=hop_length, win_length=win_length)
            s = stft_magnitude(target_wav, device=device, n_fft=n_fft, hop_length=hop_length, win_length=win_length)

            y_abs = y.abs().clamp_min(1e-12).pow(self.gamma)
            s_abs = s.abs().clamp_min(1e-12).pow(self.gamma)

            loss = loss + F.mse_loss(y_abs, s_abs) * self.factor_mag
            
            if self.factor_complex > 0:
                y_c = y_abs * torch.exp(1j * safe_angle(y))
                s_c = s_abs * torch.exp(1j * safe_angle(s))
                loss = loss + F.mse_loss(torch.view_as_real(y_c), torch.view_as_real(s_c)) * self.factor_complex

            return loss / len(self.n_ffts)

class DeepFilterNetCombinedLoss(nn.Module):
    def __init__(
        self,
        filterbank: Tensor,
        df_bins: int = 96,
        lambda_spec: float = 1.0,
        lambda_mask: float = 0.5,
        lambda_stage1: float = 0.5,
        lambda_alpha: float = 0.05,
        lambda_mrsl: float = 0.0,
        n_fft: int = 512,
        hop_length: int = 128,
        win_length: int = 512,
    ):
        super().__init__()
        self.spectral_loss = EnhancedSpectralLoss(factor_under=1.0)
        self.mask_loss = ERBMaskLoss(power_compress=0.6, factor_under=2.0)
        self.lsnr_target = LocalSnrTarget(df_bins=df_bins)
        self.alpha_loss = DfAlphaLoss(factor=lambda_alpha)

        self.lambda_stage1 = lambda_stage1
        self.lambda_spec = lambda_spec
        self.lambda_mask = lambda_mask
        self.lambda_mrsl = lambda_mrsl
        self.register_buffer("filterbank", filterbank)

        # Multi-resolution loss setup
        if lambda_mrsl > 0:
            self.mrsl = MultiResSpecLoss(n_ffts=[256, 512, 1024], gamma=0.6)
            self.n_fft = n_fft
            self.win_length = win_length
            self.hop_length = hop_length
            self.register_buffer("istft_window", torch.hann_window(win_length))
        else:
            self.mrsl = None

    def forward(self,
                final_spec: Tensor, # [B, T, F] (Y^DF)
                stage1_spec: Tensor, # [B, T, F] (Y^G)
                target_spec: Tensor, # [B, T, F] Clean target (S)
                input_spec: Tensor, # [B, T, F] Noisy input (X)
                pred_gains: Tensor, # [B, 1, T, erb_bins]
                pred_alpha: Tensor, # [B, T, 1]
               ) -> Tensor:
        l_spec = self.spectral_loss(final_spec, target_spec)

        l_stage1 = self.spectral_loss(stage1_spec, target_spec)
        
        l_mask = self.mask_loss(pred_gains, input_spec, target_spec, self.filterbank)
        
        # gt_lsnr = self.lsnr_target(target_spec, input_spec)
        # l_alpha = self.alpha_loss(pred_alpha, gt_lsnr)
        
        total_loss = self.lambda_spec * l_spec + self.lambda_mask * l_mask + self.lambda_stage1 * l_stage1
        
        l_mrsl = torch.tensor(0.0, device=final_spec.device)
        if self.mrsl is not None and self.lambda_mrsl > 0:
            enh_wav = torch.istft(
                final_spec.transpose(1, 2),
                n_fft=self.n_fft,
                hop_length=self.hop_length,
                win_length=self.win_length,
                window=self.istft_window,
            )
            
            clean_wav = torch.istft(
                target_spec.transpose(1, 2),
                n_fft=self.n_fft,
                hop_length=self.hop_length,
                win_length=self.win_length,
                window=self.istft_window,
            )

            l_mrsl = self.mrsl(enh_wav, clean_wav) * self.lambda_mrsl
            total_loss = total_loss + l_mrsl


        return total_loss
        
        

## Set up layers

In [ ]:
def channel_shuffle(x: Tensor, groups: int) -> Tensor:
    b, t, d = x.shape
    if d % groups != 0:
        raise ValueError("Feature dimension must be divisible by groups")

    features_per_group = d // groups
    x = x.view(b, t, groups, features_per_group)
    x = x.transpose(2, 3).contiguous()
    x = x.view(b, t, d)

    return x

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, kernel_size: Tuple[int, int] = (3, 2), stride: Tuple[int, int] = (1, 1), lookahead: int = 0) -> None:
        super().__init__()

        kt, kf = kernel_size

        if lookahead < 0 or lookahead > kt - 1:
            raise ValueError("lookahead must satisfy 0 <= lookahead <= kt - 1")

        time_left = kt - 1 - lookahead
        time_right = lookahead
        self.pad = (
            kf // 2, # left
            kf - 1 - kf // 2, # right
            time_left, # top
            time_right, # bottom
        )

        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=0,
            groups=in_channels,
            bias=False,
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        )

        self.norm = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU()
    def forward(self, x: Tensor) -> Tensor:
        x = F.pad(x, self.pad)
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.norm(x)
        x = self.act(x)
        return x
    
class SeparableTConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, scale_factor: int = 2, lookahead: int = 0) -> None:
        super().__init__()
        self.scale_factor = scale_factor

        self.conv = SeparableConv2d(in_channels, out_channels, kernel_size=(3, 2), lookahead=lookahead)

    def forward(self, x: Tensor) -> Tensor:
        x = F.interpolate(
            x, 
            scale_factor=(1, self.scale_factor), # [width, height] ~ [time, frequency]
            mode="nearest"
        )
        return self.conv(x)

class GroupedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()

        if in_features % groups:
            raise ValueError("in_features must be divisible by groups")

        if out_features % groups:
            raise ValueError("out_features must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.in_per_group = in_features // groups
        self.out_per_group = out_features // groups

        self.layers = nn.ModuleList(
            [
                nn.Linear(
                    self.in_per_group,
                    self.out_per_group,
                )
                for _ in range(groups)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, T, D]

        chunks = x.split(self.in_per_group, dim=-1)

        output_chunks = []

        for layer, chunk in zip(self.layers, chunks):
            y = layer(chunk)
            output_chunks.append(y)

        x = torch.cat(output_chunks, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRU(nn.Module):
    def __init__(self, input_size: int = 512, hidden_size: int = 512, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()
        if input_size % groups:
            raise ValueError("input_size must be divisible by groups")
        if hidden_size % groups:
            raise ValueError("hidden_size must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.input_per_group = input_size // groups
        self.hidden_per_groups = hidden_size // groups

        self.grus = nn.ModuleList(
            [
                nn.GRU(
                    input_size=self.input_per_group,
                    hidden_size=self.hidden_per_groups,
                    batch_first=True,
                )
                for _ in range(groups)
            ]
        )


    def forward(self, x: Tensor) -> Tensor:
        # [B, T, D]
        chunks = x.split(self.input_per_group, dim=-1)
        outputs = []

        for gru, chunk in zip(self.grus, chunks):
            # GRU returns [output, hidden]
            y, _ = gru(chunk)
            outputs.append(y)

        x = torch.cat(outputs, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRUStack(nn.Module):
    def __init__(self, size: int = 512, groups: int = 8, num_layers: int = 3) -> None:
        super().__init__()

        self.layers = nn.ModuleList(
            [
                GroupedGRU(
                    input_size=size,
                    hidden_size=size,
                    groups=groups,
                    shuffle=True,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        for layer in self.layers:
            x = layer(x)

        return x

class PConv(nn.Module):
    def __init__(self, channels: int = 64, out_channels: int = 64) -> None:
        super().__init__()

        self.conv = nn.Conv2d(channels, out_channels, kernel_size=1, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.conv(x)

## Implement ERB encoder, decoder and deep filter net

In [ ]:
class ERBEncoder(nn.Module):
    def __init__(self, erb_bins: int = 32, channels: int = 64, hidden_size: int = 512, groups: int = 8, conv_lookahead: int = 2) -> None:
        super().__init__()

        if erb_bins % 8:
            raise ValueError("erb_bins must be divisible by 8")

        self.erb_bins = erb_bins
        self.channels = channels
        self.conv_lookahead = conv_lookahead
        lookahead0 = 1 if conv_lookahead > 0 else 0
        lookahead1 = 1 if conv_lookahead > 1 else 0
        lookahead2 = 1 if conv_lookahead > 2 else 0
        self.conv0 = SeparableConv2d(1, channels, lookahead=lookahead0)
        self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=lookahead1) # B -> B / 2
        self.conv2 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=lookahead2)
        self.conv3 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=0)
        self.glinear = GroupedLinear(in_features=channels*erb_bins//8, out_features=hidden_size, groups=groups)
        self.gru = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=3)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
        x0 = self.conv0(x)
        x1 = self.conv1(x0)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)

        batch, channels, time, freq = x3.shape
        x = x3.permute(0, 2, 1, 3)
        x = x.reshape(batch, time, channels*freq)

        x = self.glinear(x)
        embedding = self.gru(x)
        embedding = self.dropout(embedding)

        return x0, x1, x2, x3, embedding

class ERBDecoder(nn.Module):
    def __init__(self, channels: int = 64, erb_bins: int = 32, hidden_size: int = 512) -> None:
        super().__init__()

        bottleneck_freq = erb_bins // 8
        bottleneck_size = channels * bottleneck_freq

        self.channels = channels
        self.bottelneck_freq = bottleneck_freq

        self.linear = GroupedLinear(hidden_size, bottleneck_size)

        self.p3 = PConv(channels)
        self.p2 = PConv(channels)
        self.p1 = PConv(channels)
        self.p0 = PConv(channels)

        self.up3 = SeparableTConv2d(channels, channels, lookahead=0)
        self.up2 = SeparableTConv2d(channels, channels, lookahead=0)
        self.up1 = SeparableTConv2d(channels, channels, lookahead=0)

        self.final_conv = SeparableConv2d(channels, 1, kernel_size=(3, 2), lookahead=0)
        self.output_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, embedding: Tensor, x0: Tensor, x1: Tensor, x2: Tensor, x3: Tensor) -> Tensor:
        batch, time, _ = embedding.shape

        x = self.linear(embedding)
        x = self.dropout(x)

        x = x.reshape(batch, time, self.channels, self.bottelneck_freq)
        x = x.permute(0, 2, 1, 3).contiguous()

        p3_x3 = self.p3(x3)
        x = x + p3_x3

        x = self.up3(x)
        x = x + self.p2(x2)

        x = self.up2(x)
        x = x + self.p1(x1)

        x = self.up1(x)
        x = x + self.p0(x0)

        gains = self.final_conv(x)
        gains = self.output_activation(gains)

        return gains

class DFNet(nn.Module):
    def __init__(self, df_bins: int, df_order: int, channels: int=64, hidden_size: int = 512, groups: int=8) -> None:
        super().__init__()

        self.df_bins = df_bins
        self.df_order = df_order
        self.channels = channels

        self.conv0 = SeparableConv2d(2, channels, lookahead=0)
        self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=0)
        
        self.projection = GroupedLinear(channels*(df_bins // 2), hidden_size, groups=groups)
        self.grus = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=2)
        self.pconv = PConv(channels=channels, out_channels=df_order*2)
        
        self.output_coefs = nn.Linear(hidden_size, df_bins*df_order*2)
        self.output_alpha = nn.Sequential(nn.Linear(hidden_size, 1), nn.Sigmoid())
        nn.init.constant_(self.output_alpha[0].bias, 2.0)
        
        self.merge = nn.Linear(hidden_size*2, hidden_size)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, complex_features: Tensor, encoder_embedding: Tensor) -> Tuple[Tensor, Tensor]:
        # x0: [B, C, T, F_DF]
        x0 = self.conv0(complex_features)

        # pconv(x0): [B, df_order * 2, T, F_DF] -> permute to [B, T, F_DF, df_order * 2]
        skip = self.pconv(x0).permute(0, 2, 3, 1)
        x = self.conv1(x0)

        batch, channels, time, freq = x.shape
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(batch, time, channels * freq)

        x = self.projection(x)
        x = self.grus(x)
        x = self.dropout(x)

        x = torch.cat([x, encoder_embedding], dim=-1)
        x = self.merge(x)

        coefficients = self.output_coefs(x)
        coefficients = coefficients.reshape(batch, time, self.df_bins, self.df_order * 2)
        coefficients = coefficients + skip
        coefficients = coefficients.reshape(batch, time, self.df_bins, self.df_order, 2)
        
        alpha = self.output_alpha(x)  # [B, T, 1]

        return coefficients, alpha

        

## Deep Neural Network (DNN)

In [ ]:
class DeepFilterNetDNN(nn.Module):
    """
    rb_features:
        [B, 1, T, B_erb]

    complex_features:
        [B, 1, T, F_df]

    erb_gains:
        [B, 1, T, B_erb]
    
    df_coefficients:
        [B, T, F_df, N, 2]
    """

    def __init__(self, erb_bins: int = 32, df_bins: int = 96, df_order: int = 5, channels: int = 64, hidden_size: int = 512, groups: int = 8, conv_lookahead: int = 2) -> None:
        super().__init__()
        
        self.encoder = ERBEncoder(
            erb_bins=erb_bins,
            channels=channels,
            hidden_size=hidden_size,
            groups=groups,
            conv_lookahead=conv_lookahead,
        )

        self.decoder = ERBDecoder(
            channels=channels,
            erb_bins=erb_bins,
            hidden_size=hidden_size
        )

        self.df_net = DFNet(
            df_bins=df_bins,
            df_order=df_order,
            channels=channels,
            hidden_size=hidden_size,
            groups=groups,
        )

    def forward(self, erb_features: Tensor, complex_features: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
        # Stage 1:
        e0, e1, e2, e3, embedding = self.encoder(erb_features)
        gains = self.decoder(embedding, e0, e1, e2, e3)

        # Stage 2:
        df_coefficients, alpha = self.df_net(complex_features, embedding)

        return gains, df_coefficients, alpha

## Build Dataset Object

In [ ]:
class IndexedERBDataset(Dataset):
    def __init__(self, index_path: str | Path, *, segment_frames: Optional[int] = 256, random_crop: bool = True) -> None:
        index_path = Path(index_path)
        self.data_dir = index_path.parent
        self.rows = [
            json.loads(line)
            for line in Path(index_path).read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        self.segment_frames = segment_frames
        self.random_crop = random_crop
        self.cached_shard_file = None
        self.cached_shard = None

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> Dict[str, Tensor]:
        row = self.rows[index]
        shard_file = row["shard_file"]

        if self.cached_shard_file != shard_file:
            shard_path = self.data_dir / shard_file
            self.cached_shard = torch.load(shard_path, map_location="cpu", weights_only=True)
            self.cached_shard_file = shard_file
        record = self.cached_shard[row["shard_index"]]
        total_frames = record["input_erb"].shape[0]
        frames = self.segment_frames or total_frames

        if total_frames >= frames:
            if self.random_crop:
                start = torch.randint(total_frames - frames + 1, ()).item()
            else:
                start = (total_frames - frames) // 2
            stop = start + frames
            result = {key: value[start:stop] for key, value in record.items()}
        else:
            result = {
                key: torch.nn.functional.pad(
                    value,
                    (0, 0, 0, frames - total_frames),
                )
                for key, value in record.items()
            }
        # Model input: [B, 1, T, E]; DataLoader adds B.
        result["input_erb"] = result["input_erb"].unsqueeze(0)
        result["target_erb"] = result["target_erb"].unsqueeze(0)
        return result

## Split Dataset

In [ ]:
class DatasetSpliter(Dataset):
    def __init__(self, dataset: Dataset, indices: Sequence[int]):
        self.dataset = dataset
        self.indices = list(indices)
        
        # For Batch Sampler
        self.rows = [
            dataset.rows[index]
            for index in indices
        ]
        
    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        original_index = self.indices[index]
        return self.dataset[original_index]

In [ ]:
def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15, seed=1234):
    generator = torch.Generator()
    generator.manual_seed(seed)
    all_indices = torch.randperm(len(dataset), generator=generator).tolist()
    
    train_end = int(len(dataset) * train_ratio)
    val_end = train_end + int(len(dataset) * val_ratio)
    train_indices = all_indices[:train_end]
    val_indices = all_indices[train_end : val_end]
    test_indices = all_indices[val_end:]

    train_dataset = DatasetSpliter(dataset, train_indices)
    val_dataset = DatasetSpliter(dataset, val_indices)
    test_dataset = DatasetSpliter(dataset, test_indices)
    return (train_dataset, val_dataset, test_dataset)
    

## BatchSampler

In [ ]:
class ShardBatchSampler(BatchSampler):
    def __init__(self, dataset, batch_size: int, drop_last: bool = False, seed: int = 1234):
        self.dataset = dataset
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.seed = seed
        self.epoch = 0
        self.indices_by_shard = defaultdict(list)
        for index, row in enumerate(dataset.rows):
            self.indices_by_shard[row["shard_file"]].append(index)
        self.shard_files = list(self.indices_by_shard.keys())

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __iter__(self) -> Iterator[List[int]]:
        generator = torch.Generator()
        generator.manual_seed(self.seed + self.epoch)
        # Shuffle order of shard
        shard_order = torch.randperm(len(self.shard_files), generator=generator).tolist()
        for shard_position in shard_order:
            shard_file = self.shard_files[shard_position]
            shard_indices = self.indices_by_shard[shard_file]
            sample_order = torch.randperm(len(shard_indices), generator=generator).tolist()
            shuffled_indices = []
            for i in sample_order:
                shuffled_indices.append(shard_indices[i])
            # Create batches from a single shard
            for start in range(0, len(shuffled_indices), self.batch_size):
                batch = shuffled_indices[start:start + self.batch_size]
                if len(batch) < self.batch_size and self.drop_last:
                    continue
                yield batch

    def __len__(self) -> int:
        total_batches = 0

        for shard_indices in self.indices_by_shard.values():
            shard_length = len(shard_indices)
            if self.drop_last:
                total_batches += shard_length // self.batch_size
            else:
                total_batches += (shard_length + self.batch_size - 1) // self.batch_size
                
        return total_batches
            

## Define Parameters

In [ ]:
sample_rate = 16000
n_fft = 512
hop_length = 128
win_length = 512
erb_bins = 32
batch_size = 20
epochs = 30
learning_rate = 1e-3
df_bins = 96
df_order = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
filterbank = make_erb_filterbank(sample_rate=sample_rate, n_fft=n_fft, erb_bins=erb_bins)
criterion = DeepFilterNetCombinedLoss(filterbank=filterbank, df_bins=df_bins, lambda_mrsl=0.1, n_fft=n_fft, hop_length=hop_length, win_length=win_length).to(device)


## Define Dataloader

In [ ]:
!ls /content/drive/MyDrive/DNS_Data/erb_store/test/

In [ ]:
idx_train_path = "/kaggle/input/datasets/quanninhhoang/erb-dns-speech-dataset/train/index.jsonl"
idx_val_path = "/kaggle/input/datasets/quanninhhoang/erb-dns-speech-dataset/val/index.jsonl"
idx_test_path = "/kaggle/input/datasets/quanninhhoang/erb-dns-speech-dataset/test/index.jsonl"
# idx_train_path = "/content/drive/MyDrive/DNS_Data/erb_store/train/index.jsonl"
# idx_val_path = "/content/drive/MyDrive/DNS_Data/erb_store/val/index.jsonl"
# idx_test_path = "/content/drive/MyDrive/DNS_Data/erb_store/test/index.jsonl"

train_dataset = IndexedERBDataset(index_path=idx_train_path)
val_dataset = IndexedERBDataset(index_path=idx_val_path)
test_dataset = IndexedERBDataset(index_path=idx_test_path, segment_frames=None)

train_batch_sampler = ShardBatchSampler(
    train_dataset,
    batch_size=batch_size,
    drop_last=False,
    seed=1234,
)

val_batch_sampler = ShardBatchSampler(
    val_dataset,
    batch_size=batch_size,
    drop_last=False,
    seed=5678,
)

test_batch_sampler = ShardBatchSampler(
    test_dataset,
    batch_size=batch_size,
    drop_last=False,
    seed=9012,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    num_workers=0,
    pin_memory=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_sampler=val_batch_sampler,
    num_workers=1,
    pin_memory=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_sampler=test_batch_sampler,
    num_workers=1,
    pin_memory=False,
)


In [ ]:
# indx_path = "/content/drive/MyDrive/DNS_Data/erb_store/index.jsonl"
# dataset = IndexedERBDataset(index_path=indx_path)

# train_dataset, val_dataset, test_dataset = split_dataset(
#     dataset,
#     train_ratio=0.70,
#     val_ratio=0.15,
#     seed=1234,
# )

# train_batch_sampler = ShardBatchSampler(
#     train_dataset,
#     batch_size=batch_size,
#     drop_last=False,
#     seed=1234,
# )

# val_batch_sampler = ShardBatchSampler(
#     val_dataset,
#     batch_size=batch_size,
#     drop_last=False,
#     seed=5678,
# )

# test_batch_sampler = ShardBatchSampler(
#     test_dataset,
#     batch_size=batch_size,
#     drop_last=False,
#     seed=9012,
# )

# train_loader = DataLoader(
#     train_dataset,
#     batch_sampler=train_batch_sampler,
#     num_workers=0,
#     pin_memory=False,
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_sampler=val_batch_sampler,
#     num_workers=1,
#     pin_memory=False,
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_sampler=test_batch_sampler,
#     num_workers=1,
#     pin_memory=False,
# )

## Generate erb band data and index files

In [ ]:
input_dir = Path("/content/drive/MyDrive/DNS_Data/synthesized/test/noisy")

target_dir = Path("/content/drive/MyDrive/DNS_Data/synthesized/test/clean")


build_erb_store(input_dir=input_dir, target_dir=target_dir, output_dir="/content/erb_store/test", 
sample_rate=sample_rate, n_fft=n_fft, hop_length=hop_length, win_length=win_length, erb_bins=erb_bins, device=device)

## Upload to Google Drive

In [ ]:
import shutil
from pathlib import Path

src_dir = Path("/content/erb_store/test")
dst_dir = Path("/content/drive/MyDrive/DNS_Data/erb_store/test")
dst_dir.mkdir(parents=True, exist_ok=True)

files = sorted(list(src_dir.glob("shard_*.pt")) + list(src_dir.glob("*.json*")))

print(f"Total files to transfer: {len(files)}")

for i, file in enumerate(files, 1):
    target = dst_dir / file.name
    # Skip if file already copied and file size matches
    if target.exists() and target.stat().st_size == file.stat().st_size:
        print(f"[{i}/{len(files)}] Already exists: {file.name}")
        continue

    print(f"[{i}/{len(files)}] Copying {file.name} ({file.stat().st_size / (1024**2):.1f} MB)...", flush=True)
    shutil.copy2(file, target)

print("\nSuccessfully copied all files to Google Drive!")

## Validation Function

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, synthesis_matrix, device):
    model.eval()
    total_loss = 0.0
    total_batches = 0

    for batch in loader:
        input_erb = batch["input_erb"].to(device, dtype=torch.float32, non_blocking=True)
        input_spec = batch["input_spec"].to(device, non_blocking=True)
        target_spec = batch["target_spec"].to(device, non_blocking=True)
        complex_in = torch.stack([input_spec[:, :, :df_bins].real, input_spec[:, :, :df_bins].imag], dim=1)
        
        predicted_gains, coefficients, alpha = model(input_erb, complex_in)
        
        stage1_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)
        final_spec = apply_deep_filter(stage1_spec=stage1_spec, coefficients=coefficients, alpha=alpha, df_bins=df_bins, df_order=df_order)
        
        loss = criterion(
                final_spec=final_spec,
                stage1_spec=stage1_spec,
                target_spec=target_spec,
                input_spec=input_spec,
                pred_gains=predicted_gains,
                pred_alpha=alpha,
            )
        total_loss += loss.item()
        total_batches += 1
    return total_loss / max(total_batches, 1)


## Training Step

In [ ]:
def train_model(model, train_loader, val_loader, train_batch_sampler, synthesis_matrix, optimizer, scheduler, epochs, checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_val_loss = float("inf")
    history = []

    for epoch in range(epochs):
        epoch_start_time = time.perf_counter()
        
        train_batch_sampler.set_epoch(epoch)
        model.train()

        total_train_loss = 0.0
        total_batches = 0

        for batch in train_loader:
            input_erb = batch["input_erb"].to(device, dtype=torch.float32, non_blocking=True)
            input_spec = batch["input_spec"].to(device, non_blocking=True)
            target_spec = batch["target_spec"].to(device, non_blocking=True)

            # Prepare 2-channel complex input (Real, Imag) for DFNet
            complex_in = torch.stack([input_spec[:, :, :df_bins].real, input_spec[:, :, :df_bins].imag], dim=1)
            
            optimizer.zero_grad(set_to_none=True)

            # 1. Model parameter estimation
            gains, coefficients, alpha = model(input_erb, complex_in)

            # 2. Stage 1: ERB envelope filtering
            stage1_spec = apply_erb_gains(input_spec, gains, synthesis_matrix)

            # 3. Stage 2: Deep filtering with alpha weighting
            final_spec = apply_deep_filter(stage1_spec=stage1_spec, coefficients=coefficients, alpha=alpha, df_bins=df_bins, df_order=df_order)

            # 4. Loss computation
            loss = criterion(
                final_spec=final_spec,
                stage1_spec=stage1_spec,
                target_spec=target_spec,
                input_spec=input_spec,
                pred_gains=gains,
                pred_alpha=alpha,
            )
            
            loss.backward()
            optimizer.step()
            total_train_loss += loss.detach().item()
            total_batches += 1
            
        train_loss = (total_train_loss / max(total_batches, 1))
        
        val_loss = evaluate_model(
            model=model,
            loader=val_loader,
            synthesis_matrix=synthesis_matrix,
            device=device,
        )

        scheduler.step(val_loss)

        epoch_duration = time.perf_counter() - epoch_start_time
        mins, secs = divmod(int(epoch_duration), 60)
        time_str = f"{mins:02d}m {secs:02d}s"
        
        history.append(
            {
                "epoch": epoch+1,
                "train_loss": train_loss,
                "val_loss": val_loss,
            }
        )
        epoch_checkpoint = checkpoint_dir / (f"epoch_{epoch + 1:03d}.ckpt")
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
            },
            epoch_checkpoint,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_checkpoint = checkpoint_dir / (f"best_model.ckpt")
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                },
                best_checkpoint,
            )
            print(
                f"Epoch {epoch + 1:02d}/{epochs:02d} [{time_str}] "
                f"train={train_loss:.6f} "
                f"val={val_loss:.6f} "
                f"best checkpoint saved",
                flush=True,
            )
        else:
            print(
                f"Epoch {epoch + 1:02d}/{epochs:02d} [{time_str}] "
                f"train={train_loss:.6f} "
                f"val={val_loss:.6f}",
                flush=True,
            )

    return history
            

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

model = DeepFilterNetDNN(erb_bins=erb_bins).to(device)
synthesis_matrix = erb_synthesis_matrix(filterbank).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)


In [ ]:
checkpoint_dir = "/kaggle/working/erb_checkpoints"

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    train_batch_sampler=train_batch_sampler,
    synthesis_matrix=synthesis_matrix,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
    checkpoint_dir=checkpoint_dir,
)

## Visualize Training and Validation Loss

In [ ]:
epochs_plot = [
    item["epoch"]
    for item in history
]

train_loss = [
    item["train_loss"]
    for item in history
]

val_loss = [
    item["val_loss"]
    for item in history
]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_plot,
    train_loss,
    color="red",
    marker="o",
    label="Training Loss"
)
plt.plot(
    epochs_plot,
    val_loss,
    color="blue",
    marker="s",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Testing Step - Compute standard speech metrices (PESQ, STOI, SI-SDR)

### 1. Using Standard Speech Metrices (PESQ, STOI, SI-SDR)

In [ ]:
try:
    import pystoi
except ImportError:
    pystoi = None

try:
    from pesq import pesq
except ImportError:
    pesq = None

def spec_to_wav(spec: Tensor, n_fft: int = 512, hop_length: int = 128, win_length: int = 512, window: Optional[Tensor] = None) -> Tensor:
    """Convert complex spectrogram [B, T, F] back to time-domain waveform [B, N-sampels]"""
    if window is None:
        window = torch.hann_window(win_length, device=spec.device)

    # torch.istft expects shape [B, F, T]
    spec_transposed = spec.transpose(-1, -2).contiguous()
    wav = torch.istft(spec_transposed, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=window, center=True)
    return wav

def calculate_si_sdr(estimate: Tensor, reference: Tensor, eps: float = 1e-8) -> float:
    est = estimate.flatten().double()
    ref = reference.flatten().double()

    # Zero-mean
    est = est - est.mean()
    ref = ref - ref.mean()

    # Projection scale factor alpha 
    ref_energy = torch.sum(ref ** 2) + eps
    alpha = torch.sum(est * ref) / ref_energy

    e_target = alpha * ref
    e_res = est - e_target

    target_pow = torch.sum(e_target ** 2) + eps
    res_pow = torch.sum(e_res ** 2) + eps

    return float(10 * torch.log10(target_pow / res_pow).item())

def calculate_stoi(clean_wav: np.ndarray, deg_wav: np.ndarray, sr: int = 16000) -> float:
    if pystoi is None:
        return float("nan")
    return float(pystoi.stoi(clean_wav, deg_wav, sr, extended=False))

def calculate_pesq(clean_wav: np.ndarray, deg_wav: np.ndarray, sr: int = 16000) -> float:
    if pesq is None:
        return float("nan")
    try:
        c = np.clip(clean_wav.astype(np.float32), -1.0, 1.0)
        d = np.clip(deg_wav.astype(np.float32), -1.0, 1.0)
        mode = "wb" if sr in (16000, 48000) else "nb"
        return float(pesq(sr, c, d, mode))
    except Exception:
        return float("nan")


@torch.inference_mode()
def evaluate_speech_metrices(model: nn.Module, test_loader, synthesis_matrix: Tensor, device: torch.device, df_bins: int = 96, df_order: int = 5, sample_rate: int = 16000,
                            n_fft: int = 512, hop_length: int = 128, win_length: int = 512, max_samples: int = 100) -> Dict[str, float]:
    model.eval()
    synthesis_matrix = synthesis_matrix.to(device)
    window = torch.hann_window(win_length, device=device)

    metrices = {
        "pesq_noisy": [], "pesq_stage1": [], "pesq_final": [],
        "stoi_noisy": [], "stoi_stage1": [], "stoi_final": [],
        "sisdr_noisy": [], "sisdr_stage1": [], "sisdr_final": [],
    }


    count = 0
    print(f"Starting objective evaluation on test set (SR: {sample_rate} Hz)...")

    for batch in test_loader:
        input_erb = batch["input_erb"].to(device, dtype=torch.float32, non_blocking=True)
        input_spec = batch["input_spec"].to(device, non_blocking=True)
        target_spec = batch["target_spec"].to(device, non_blocking=True)

        complex_in = torch.stack([input_spec[:, :, :df_bins].real, input_spec[:, :, :df_bins].imag], dim=1)

        predicted_gains, coefficients, alpha = model(input_erb, complex_in)

        stage1_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)

        final_spec = apply_deep_filter(
            stage1_spec=stage1_spec,
            coefficients=coefficients,
            alpha=alpha,
            df_bins=df_bins,
            df_order=df_order,
        )

        noisy_wav = spec_to_wav(input_spec, n_fft, hop_length, win_length, window)
        clean_wav = spec_to_wav(target_spec, n_fft, hop_length, win_length, window)
        final_wav = spec_to_wav(final_spec, n_fft, hop_length, win_length, window)
        stage1_wav = spec_to_wav(stage1_spec, n_fft, hop_length, win_length, window)

        for i in range(clean_wav.shape[0]):
            c_wav = clean_wav[i]
            n_wav = noisy_wav[i]
            f_wav = final_wav[i]
            s1_wav = stage1_wav[i]

            """SI-SDR"""
            metrices["sisdr_noisy"].append(calculate_si_sdr(n_wav, c_wav))
            metrices["sisdr_final"].append(calculate_si_sdr(f_wav, c_wav))
            metrices["sisdr_stage1"].append(calculate_si_sdr(s1_wav, c_wav))

            """PESQ & STOI"""
            c_np = c_wav.cpu().numpy()
            n_np = n_wav.cpu().numpy()
            f_np = f_wav.cpu().numpy()
            s1_np = s1_wav.cpu().numpy()

            # Align lengths
            min_len = min(len(c_np), len(n_np), len(s1_np), len(f_np))
            c_np, n_np, f_np, s1_np = c_np[:min_len], n_np[:min_len], f_np[:min_len], s1_np[:min_len]

            metrices["stoi_noisy"].append(calculate_stoi(c_np, n_np, sample_rate))
            metrices["stoi_final"].append(calculate_stoi(c_np, f_np, sample_rate))
            metrices["stoi_stage1"].append(calculate_stoi(c_np, s1_np, sample_rate))


            p_n = calculate_pesq(c_np, n_np, sample_rate)
            p_f = calculate_pesq(c_np, f_np, sample_rate)
            p_s1 = calculate_pesq(c_np, s1_np, sample_rate)

            if not np.isnan(p_n):
                metrices["pesq_noisy"].append(p_n)
            if not np.isnan(p_f):
                metrices["pesq_final"].append(p_f)
            if not np.isnan(p_s1):
                metrices["pesq_stage1"].append(p_s1)

            count += 1
            if max_samples and count >= max_samples:
                break
        if max_samples and count >= max_samples:
            break

    summary = {
        # PESQ
        "PESQ Noisy": np.nanmean(metrices["pesq_noisy"]),
        "PESQ Stage 1 (ERB)": np.nanmean(metrices["pesq_stage1"]),
        "PESQ Final (Stage 2)": np.nanmean(metrices["pesq_final"]),
        "Delta-PESQ": np.nanmean(metrices["pesq_final"]) - np.nanmean(metrices["pesq_noisy"]),
        
        # STOI
        "STOI Noisy": np.nanmean(metrices["stoi_noisy"]),
        "STOI Stage 1 (ERB)": np.nanmean(metrices["stoi_stage1"]),
        "STOI Final (Stage 2)": np.nanmean(metrices["stoi_final"]),
        "Delta-STOI": np.nanmean(metrices["stoi_final"]) - np.nanmean(metrices["stoi_noisy"]),
        
        # SI-SDR (Fixed keys)
        "SI-SDR Noisy (dB)": np.nanmean(metrices["sisdr_noisy"]),
        "SI-SDR Stage 1 (ERB)": np.nanmean(metrices["sisdr_stage1"]),
        "SI-SDR Final (Stage 2)": np.nanmean(metrices["sisdr_final"]),
        "Delta-SI-SDR (dB)": np.nanmean(metrices["sisdr_final"]) - np.nanmean(metrices["sisdr_noisy"]),
    }
    
    return summary

In [ ]:
checkpoint_path = "/kaggle/working/erb_checkpoints/best_model.ckpt"
# checkpoint_path = "/content/drive/MyDrive/DNS_Data/erb_checkpoints/best_model.ckpt"
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

model.load_state_dict(checkpoint["model_state_dict"])

summary = evaluate_speech_metrices(
    model=model,
    test_loader=test_loader,
    synthesis_matrix=synthesis_matrix,
    device=device,
    sample_rate=16000,
    max_samples=150,
)

print(os.linesep + "=" * 75)
print(f"{'Metric':<20} | {'Noisy':<10} | {'Stage 1 (ERB)':<13} | {'Final (Stage 2)':<15} | {'Delta (Δ)':<10}")
print("-" * 75)
# PESQ
print(
    f"{'PESQ (1.0 - 4.5)':<20} | "
    f"{summary['PESQ Noisy']:<10.2f} | "
    f"{summary['PESQ Stage 1 (ERB)']:<13.2f} | "
    f"{summary['PESQ Final (Stage 2)']:<15.2f} | "
    f"{summary['Delta-PESQ']:<+10.2f}"
)
# STOI
print(
    f"{'STOI (0.0 - 1.0)':<20} | "
    f"{summary['STOI Noisy']:<10.3f} | "
    f"{summary['STOI Stage 1 (ERB)']:<13.3f} | "
    f"{summary['STOI Final (Stage 2)']:<15.3f} | "
    f"{summary['Delta-STOI']:<+10.3f}"
)
# SI-SDR
print(
    f"{'SI-SDR (dB)':<20} | "
    f"{summary['SI-SDR Noisy (dB)']:<10.2f} | "
    f"{summary['SI-SDR Stage 1 (ERB)']:<13.2f} | "
    f"{summary['SI-SDR Final (Stage 2)']:<15.2f} | "
    f"{summary['Delta-SI-SDR (dB)']:<+10.2f}"
)
print("=" * 75 + os.linesep)

### 2. Using error computation

In [ ]:
def spectrogram_similarity(predicted_spec, target_spec):
    error = (predicted_spec - target_spec).abs().square().flatten(1).sum(dim=1)
    target_energy = (target_spec.abs().square().flatten(1).sum(dim=1))
    normalized_error = error / (target_energy + 1e-8)
    similarity = (1.0 - normalized_error).clamp(0.0, 1.0)
    return similarity * 100.0
    

In [ ]:
checkpoint_path = "/kaggle/working/erb_checkpoints/best_model.ckpt"
#checkpoint_path = "/content/drive/MyDrive/DNS_Data/erb_checkpoints/best_model.ckpt"
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

In [ ]:
total_similarity = 0.0
total_accuracy_stage1 = 0.0
total_samples = 0
total_stage1_samples = 0

with torch.inference_mode():
    for batch in test_loader:
        input_erb = batch["input_erb"].to(device, dtype=torch.float32, non_blocking=True)
        input_spec = batch["input_spec"].to(device, non_blocking=True)
        target_spec = batch["target_spec"].to(device, non_blocking=True)

        complex_in = torch.stack([input_spec[:, :, :df_bins].real, input_spec[:, :, :df_bins].imag], dim=1)
        predicted_gains, coefficients, alpha = model(input_erb, complex_in)
        
        stage1_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)

        final_spec = apply_deep_filter(
            stage1_spec=stage1_spec,
            coefficients=coefficients,
            alpha=alpha,
            df_bins=df_bins,
            df_order=df_order,
        )

        similarity_stage1 = spectrogram_similarity(stage1_spec, target_spec)
        total_accuracy_stage1 += similarity_stage1.sum().item()
        total_stage1_samples += similarity_stage1.numel()
        
        similarity_final = spectrogram_similarity(final_spec, target_spec)
        total_similarity += similarity_final.sum().item()
        total_samples += similarity_final.numel()

average_similarity = (total_similarity / max(total_samples, 1))
average = (total_accuracy_stage1 / max(total_stage1_samples, 1))
print(f"Accuracy with Stage 2: {average_similarity:.2f}%")
print(f"Accuracy without Stage 2: {average:.2f}%")

## Listen a single enhanced audio

In [ ]:
from IPython.display import Audio, display
checkpoint_path = "/kaggle/working/erb_checkpoints/best_model.ckpt"
#checkpoint_path = "/content/drive/MyDrive/DNS_Data/erb_checkpoints/best_model.ckpt" 
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

batch = next(iter(test_loader))

input_erb = batch["input_erb"].to(device, dtype=torch.float32)
input_spec = batch["input_spec"].to(device)
target_spec = batch["target_spec"].to(device)
complex_in = torch.stack([input_spec[:, :, :df_bins].real, input_spec[:, :, :df_bins].imag], dim=1)

with torch.no_grad():
    predicted_gains, coefficients, alpha = model(input_erb, complex_in)

    stage1_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)

    final_spec = apply_deep_filter(
        stage1_spec=stage1_spec,
        coefficients=coefficients,
        alpha=alpha,
        df_bins=df_bins,
        df_order=df_order,
    )


noisy_wav = spec_to_wav(input_spec, n_fft=512, hop_length=128, win_length=512)
enh_wav = spec_to_wav(final_spec, n_fft=512, hop_length=128, win_length=512)
clean_wav = spec_to_wav(target_spec, n_fft=512, hop_length=128, win_length=512)

noisy_audio = noisy_wav[0].cpu()
clean_audio = clean_wav[0].cpu()
enh_audio = enh_wav[0].cpu()

In [ ]:
print("\nNoisy Audio (input):")
display(Audio(noisy_audio.numpy(), rate=16000))

print("\nEnhanced Audio (Model Output):")
display(Audio(enh_audio.numpy(), rate=16000))

print("\nClean Audio (Target):")
display(Audio(clean_audio.numpy(), rate=16000))

## Check whether GPU is available or not

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")